In [ ]:
from pathlib import Path
import shutil
import yaml

In [ ]:
def create_filtered_css_dataset_for_combined_training(
    source_dataset_dir,
    target_dataset_dir="construction-site-safety-filtered",
    source_split_dirs=None,
    copy_images_without_remaining_labels=False,
    overwrite=True
):
    """
    Filtert den Construction Site Safety Datensatz und kopiert ihn in eine
    YOLO-kompatible Zielstruktur, die zur vorherigen Struktur passt.

    Originalklassen:
        0: Hardhat
        1: Mask
        2: NO-Hardhat
        3: NO-Mask
        4: NO-Safety Vest
        5: Person
        6: Safety Cone
        7: Safety Vest
        8: machinery
        9: vehicle

    Zielklassen:
        0: head
        1: helmet
        2: vest
        3: person

    Mapping:
        0: Hardhat     -> 1: helmet
        7: Safety Vest -> 2: vest
        5: Person      -> 3: person

    Erwartete mögliche Quellstruktur:
        source_dataset_dir/
            train/
                images/
                labels/
            valid/
                images/
                labels/
            test/
                images/
                labels/

    Zielstruktur:
        target_dataset_dir/
            images/
                train/
                val/
                test/
            labels/
                train/
                val/
                test/
            data.yaml
    """

    source_dataset_dir = Path(source_dataset_dir)
    target_dataset_dir = Path(target_dataset_dir)

    if source_split_dirs is None:
        source_split_dirs = {
            "train": "train",
            "val": "valid",
            "test": "test"
        }

    old_to_new_class_map = {
        0: 1,  # Hardhat -> helmet
        7: 2,  # Safety Vest -> vest
        5: 3   # Person -> person
    }

    new_class_names = {
        0: "head",
        1: "helmet",
        2: "vest",
        3: "person"
    }

    kept_class_names = {
        1: "helmet",
        2: "vest",
        3: "person"
    }

    image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    if not source_dataset_dir.exists():
        raise FileNotFoundError(f"Quellordner nicht gefunden: {source_dataset_dir}")

    if target_dataset_dir.exists() and overwrite:
        shutil.rmtree(target_dataset_dir)

    target_dataset_dir.mkdir(parents=True, exist_ok=True)

    stats = {
        "copied_images": 0,
        "written_labels": 0,
        "skipped_images_without_relevant_labels": 0,
        "skipped_missing_label_files": 0,
        "kept_objects": {
            "head": 0,
            "helmet": 0,
            "vest": 0,
            "person": 0
        }
    }

    for target_split, source_split in source_split_dirs.items():
        source_split_dir = source_dataset_dir / source_split

        source_images_dir = source_split_dir / "images"
        source_labels_dir = source_split_dir / "labels"

        target_images_dir = target_dataset_dir / "images" / target_split
        target_labels_dir = target_dataset_dir / "labels" / target_split

        target_images_dir.mkdir(parents=True, exist_ok=True)
        target_labels_dir.mkdir(parents=True, exist_ok=True)

        if not source_images_dir.exists():
            print(f"Warnung: Bildordner nicht gefunden: {source_images_dir}")
            continue

        if not source_labels_dir.exists():
            print(f"Warnung: Labelordner nicht gefunden: {source_labels_dir}")
            continue

        image_paths = [
            p for p in source_images_dir.iterdir()
            if p.suffix.lower() in image_extensions
        ]

        for image_path in image_paths:
            label_path = source_labels_dir / f"{image_path.stem}.txt"

            if not label_path.exists():
                stats["skipped_missing_label_files"] += 1

                if copy_images_without_remaining_labels:
                    shutil.copy2(image_path, target_images_dir / image_path.name)
                    (target_labels_dir / f"{image_path.stem}.txt").write_text("")
                    stats["copied_images"] += 1
                    stats["written_labels"] += 1

                continue

            new_label_lines = []

            with open(label_path, "r", encoding="utf-8") as f:
                lines = f.readlines()

            for line in lines:
                line = line.strip()

                if not line:
                    continue

                parts = line.split()

                old_class_id = int(parts[0])

                if old_class_id not in old_to_new_class_map:
                    continue

                new_class_id = old_to_new_class_map[old_class_id]

                # YOLO-Format bleibt:
                # class x_center y_center width height
                new_line = " ".join([str(new_class_id)] + parts[1:])
                new_label_lines.append(new_line)

                class_name = kept_class_names[new_class_id]
                stats["kept_objects"][class_name] += 1

            if not new_label_lines and not copy_images_without_remaining_labels:
                stats["skipped_images_without_relevant_labels"] += 1
                continue

            shutil.copy2(image_path, target_images_dir / image_path.name)
            stats["copied_images"] += 1

            target_label_path = target_labels_dir / f"{image_path.stem}.txt"

            with open(target_label_path, "w", encoding="utf-8") as f:
                f.write("\n".join(new_label_lines))

                if new_label_lines:
                    f.write("\n")

            stats["written_labels"] += 1

    data_yaml = {
        "path": str(target_dataset_dir),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": new_class_names
    }

    yaml_path = target_dataset_dir / "data.yaml"

    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(data_yaml, f, sort_keys=False)

    print("Fertig.")
    print(f"Zielordner: {target_dataset_dir}")
    print(f"Kopierte Bilder: {stats['copied_images']}")
    print(f"Geschriebene Label-Dateien: {stats['written_labels']}")
    print(f"Übersprungene Bilder ohne relevante Labels: {stats['skipped_images_without_relevant_labels']}")
    print(f"Fehlende Label-Dateien: {stats['skipped_missing_label_files']}")
    print("Behaltene Objekte:")
    for name, count in stats["kept_objects"].items():
        print(f"  {name}: {count}")

    return stats

In [ ]:
stats_css = create_filtered_css_dataset_for_combined_training(
    source_dataset_dir=r"C:\DataProjects\Hackathon\2026_TUM_Science\external_datasets\Construction Site Safety Image Dataset Roboflow\css-data",
    target_dataset_dir=r"C:\DataProjects\Hackathon\2026_TUM_Science\external_datasets\construction-site-safety-filtered"
)